# One-Click Colab Runner

Use this if the whole repo is already in Google Drive.

Manual setup before running:
1. Set Colab runtime to GPU.
2. Put this repo folder in `MyDrive/Neural_Image_Caption_Generator`.
3. Either put Flickr8k in `MyDrive/Neural_Image_Caption_Generator/data/flickr8k`, or set `FLICKR8K_SOURCE_DIR` below to wherever it is in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

# Change this only if your repo folder has a different name/location.
REPO_DIR = Path('/content/drive/MyDrive/Neural_Image_Caption_Generator')

# If Flickr8k is already inside the repo at data/flickr8k, leave this blank.
# If it is somewhere else in Drive, set it, e.g. Path('/content/drive/MyDrive/flickr8k').
FLICKR8K_SOURCE_DIR = None

assert REPO_DIR.exists(), f'Repo folder not found: {REPO_DIR}'
os.chdir(REPO_DIR)
print('Working in', Path.cwd())

In [ ]:
!python -m pip install -q -r requirements.txt
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Prepare Flickr8k

If you do **not** have the dataset in Drive yet, upload your Kaggle `kaggle.json` in the next optional cell and then run the Kaggle download cell.

In [ ]:
import sys, os, csv, random as _random
from pathlib import Path

sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

data_root   = REPO_DIR / 'data' / 'flickr8k'
source_dir  = Path(FLICKR8K_SOURCE_DIR).expanduser().resolve() if FLICKR8K_SOURCE_DIR else (REPO_DIR / 'data' / 'flickr8k').resolve()
data_root.mkdir(parents=True, exist_ok=True)

# ── 1. images folder ────────────────────────────────────────────────────────
image_dir = None
for _c in source_dir.rglob('*'):
    if _c.is_dir() and _c.name.lower() in {'images', 'flickr8k_dataset', 'flicker8k_dataset'}:
        if len(list(_c.glob('*.jpg'))) > 100:
            image_dir = _c
            break
assert image_dir, f'No images folder found under {source_dir}'
img_dst = data_root / 'images'
if not img_dst.exists() and not img_dst.is_symlink():
    os.symlink(image_dir.resolve(), img_dst)
    print(f'Linked images folder: {image_dir}')

# ── 2. captions.txt ─────────────────────────────────────────────────────────
caps_src = next(source_dir.rglob('captions.txt'), None)
assert caps_src, f'captions.txt not found under {source_dir}'
caps_dst = data_root / 'captions.txt'
if not caps_dst.exists() and not caps_dst.is_symlink():
    os.symlink(caps_src.resolve(), caps_dst)
    print('Linked captions.txt')

# ── 3. Split files – link if present, otherwise auto-generate ───────────────
SPLIT_NAMES = ['Flickr_8k.trainImages.txt', 'Flickr_8k.devImages.txt', 'Flickr_8k.testImages.txt']
for _fname in SPLIT_NAMES:
    _src = next(source_dir.rglob(_fname), None)
    if _src:
        _dst = data_root / _fname
        if not _dst.exists() and not _dst.is_symlink():
            os.symlink(_src.resolve(), _dst)

if any(not (data_root / f).exists() for f in SPLIT_NAMES):
    print('Split files missing – generating from captions.txt ...')
    _names = set()
    with open(caps_dst, newline='') as _f:
        for _raw in _f:
            _line = _raw.strip()
            if not _line: continue
            if '\t' in _line:
                _img_id = _line.split('\t', 1)[0]
            else:
                _row = next(csv.reader([_line]))
                if not _row or _row[0].lower() in {'image', 'image_name', 'filename'}: continue
                _img_id = _row[0]
            _names.add(os.path.basename(_img_id.strip().split('#')[0]))
    _names = sorted(_names)
    _rng = _random.Random(42)
    _rng.shuffle(_names)
    print(f'  {len(_names)} unique images found')
    _n = len(_names)
    _n_train = 6000 if _n >= 8000 else int(_n * .75)
    _n_val   = 1000 if _n >= 8000 else int(_n * .125)
    _n_test  = 1000 if _n >= 8000 else _n - _n_train - _n_val
    for _fname, _imgs in [
        ('Flickr_8k.trainImages.txt', _names[:_n_train]),
        ('Flickr_8k.devImages.txt',   _names[_n_train:_n_train+_n_val]),
        ('Flickr_8k.testImages.txt',  _names[_n_train+_n_val:_n_train+_n_val+_n_test]),
    ]:
        _dst = data_root / _fname
        if not _dst.exists():
            _dst.write_text('\n'.join(_imgs) + '\n')
            print(f'  Generated {_fname} ({len(_imgs)} images)')

# ── 4. Validate ──────────────────────────────────────────────────────────────
from utils import validate_dataset_layout
validate_dataset_layout(str(data_root))
print('Dataset ready:', data_root)


### Optional Kaggle Download Path

Only use these two cells if Flickr8k is not already in Drive. In Kaggle, create an API token and upload `kaggle.json` here.

In [ ]:
# OPTIONAL: upload kaggle.json, then run the next cell.
# from google.colab import files
# files.upload()
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# OPTIONAL: Kaggle download. Uncomment only if you used the kaggle.json cell above.
# !python scripts/colab_setup.py --repo_dir "$REPO_DIR" --download_kaggle

## Validate Code

In [ ]:
!pytest -q

## Train

In [ ]:
!python train.py

## Evaluate

In [ ]:
!python evaluate.py \
  --checkpoint checkpoints/best.pt \
  --data_root data/flickr8k \
  --vocab data/flickr8k/vocab.json \
  --split test \
  --batch_size 64 \
  --results_out results/test_bleu.json

## Visualize Attention

In [ ]:
!python visualize.py \
  --checkpoint checkpoints/best.pt \
  --vocab data/flickr8k/vocab.json \
  --data_root data/flickr8k \
  --split test \
  --num_examples 6 \
  --output_dir results/attention_examples \
  --overlay_style paper \
  --dpi 200

In [ ]:
from IPython.display import Image, display
for path in sorted(Path('results/attention_examples').glob('*.png'))[:6]:
    print(path)
    display(Image(filename=str(path)))